In [ ]:
import matplotlib.pyplot as plt

from utils.data import DataManager
from utils.tools.config import (
    BENCHMARKS, 
    RISK_ANALYSIS,
    ANALYSIS_START_DATE,
    ANALYSIS_END_DATE
)

from utils.analysis.capm import (
    CAPMAnalyzer,
    PortfolioOptimizationAnalyzer,
    MultiAssetCAPMAnalyzer,
    CAPMReporter,
    PortfolioReporter,
    MultiAssetReporter
)

from utils.visualizations import (
    CAPMVisualizer,
    PortfolioOptimizationVisualizer,
    MultiAssetCAPMVisualizer
)

In [ ]:
# 📊 CONFIGURACIÓN DEL ANÁLISIS
# Las fechas vienen de config.py (ANALYSIS_DATES)
# Personaliza aquí solo si necesitas valores diferentes

# Portfolio a analizar
TICKERS = ["META", "AAPL", "GOOGL", "MSFT", "NVDA"]
BENCHMARK_NAME = "SP500"

# (Opcional) Fechas personalizadas - Deja vacío para usar config.py
USE_CUSTOM_DATES = False
START_DATE = ""  # Ej: "2020-01-01"
END_DATE = ""    # Ej: "2024-12-31"

# Constantes desde config
BENCHMARK_TICKER = BENCHMARKS[BENCHMARK_NAME]
RISK_FREE_RATE = RISK_ANALYSIS['risk_free_rate']
ANNUAL_FACTOR = RISK_ANALYSIS['annual_factor']

# Resolver fechas
if USE_CUSTOM_DATES and START_DATE and END_DATE:
    final_start, final_end = START_DATE, END_DATE
    print(f"📅 Usando fechas personalizadas: {final_start} → {final_end}")
else:
    final_start, final_end = ANALYSIS_START_DATE, ANALYSIS_END_DATE
    print(f"📅 Usando fechas de config.py: {final_start} → {final_end}")

print(f"\n📊 Portfolio: {len(TICKERS)} activos")
print(f"📈 Benchmark: {BENCHMARK_NAME} ({BENCHMARK_TICKER})")
print(f"💰 Risk-Free Rate: {RISK_FREE_RATE:.2%}")

In [ ]:
# 🔧 INICIALIZACIÓN
# Crear DataManager compartido
data_manager = DataManager()

# Inicializar analyzers
capm_analyzer = CAPMAnalyzer(annual_factor=ANNUAL_FACTOR)
portfolio_optimizer = PortfolioOptimizationAnalyzer(annual_factor=ANNUAL_FACTOR)
multi_capm = MultiAssetCAPMAnalyzer(annual_factor=ANNUAL_FACTOR)

# Inicializar reporters
capm_reporter = CAPMReporter(capm_analyzer)
portfolio_reporter = PortfolioReporter(portfolio_optimizer)
multi_asset_reporter = MultiAssetReporter(multi_capm)

# Inicializar visualizers
capm_viz = CAPMVisualizer(capm_analyzer)
portfolio_viz = PortfolioOptimizationVisualizer(portfolio_optimizer)
multi_asset_viz = MultiAssetCAPMVisualizer(multi_capm)

print("✅ Analyzers, reporters y visualizers inicializados")

In [ ]:
# 📥 DESCARGA DE DATOS
print("\n🔄 Descargando datos...")

assets_prices, benchmark_prices = data_manager.download_portfolio_with_benchmark(
    tickers=TICKERS,
    benchmark_name=BENCHMARK_NAME,
    start_date=final_start,
    end_date=final_end
)

# Calcular retornos
returns = assets_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

print(f"\n✅ Datos descargados:")
print(f"   Período: {assets_prices.index[0].date()} → {assets_prices.index[-1].date()}")
print(f"   Días: {len(assets_prices)}")
print(f"   Activos: {list(returns.columns)}")

In [ ]:
# 📈 ANÁLISIS CAPM INDIVIDUAL
# Analiza un activo específico vs el mercado

ASSET_TO_ANALYZE = "META"  # Cambia aquí el ticker

print(f"\n🔍 Analizando {ASSET_TO_ANALYZE} con CAPM...\n")

capm_reporter.generate_report(
    asset_returns=returns[ASSET_TO_ANALYZE].values,
    market_returns=benchmark_returns.values,
    risk_free_rate=RISK_FREE_RATE,
    asset_name=ASSET_TO_ANALYZE
)

In [ ]:
# 📊 VISUALIZACIÓN CAPM INDIVIDUAL
fig1 = capm_viz.plot_capm_analysis(
    asset_returns=returns[ASSET_TO_ANALYZE].values,
    market_returns=benchmark_returns.values,
    risk_free_rate=RISK_FREE_RATE,
    asset_name=ASSET_TO_ANALYZE
)
plt.show()

In [ ]:
# 🎯 FRONTERA EFICIENTE Y CML
print("\n🔍 Calculando frontera eficiente...\n")

fig2 = portfolio_viz.plot_efficient_frontier_analysis(
    returns=returns,
    risk_free_rate=RISK_FREE_RATE
)
plt.show()

In [ ]:
# ⭐ PORTFOLIO TANGENTE (Máximo Sharpe Ratio)
print("\n🔍 Calculando portfolio tangente...\n")

portfolio_reporter.generate_tangent_report(
    returns=returns,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# 📊 ANÁLISIS CAPM MULTI-ACTIVO (Resumen)
print("\n🔍 Analizando todos los activos con CAPM...\n")

multi_asset_reporter.generate_summary_report(
    returns=returns,
    market_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# 📋 TABLA DE RESULTADOS CAPM
capm_results = multi_capm.analyze_multiple(
    returns=returns,
    market_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

print("\n📊 Resultados detallados de CAPM:\n")
display(capm_results)

In [ ]:
# 📊 VISUALIZACIÓN MULTI-ACTIVO
fig3 = multi_asset_viz.plot_multi_asset_analysis(
    returns=returns,
    market_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)
plt.show()

In [ ]:
# 🔬 COMPARAR MÚLTIPLES ACTIVOS
# Analiza cada activo individualmente

print("\n🔬 Análisis individual de cada activo:\n")
print(f"{'Ticker':<8} {'Beta':>6} {'Alpha Anual':>12} {'R²':>6} {'Significativo'}")
print("-" * 50)

for ticker in TICKERS:
    capm_result = capm_analyzer.analyze_capm(
        asset_returns=returns[ticker].values,
        market_returns=benchmark_returns.values,
        risk_free_rate=RISK_FREE_RATE
    )
    
    sig = capm_analyzer.test_alpha_significance(
        asset_returns=returns[ticker].values,
        market_returns=benchmark_returns.values,
        risk_free_rate=RISK_FREE_RATE
    )
    
    is_sig = "✓" if sig['is_significant'] else "✗"
    
    print(f"{ticker:<8} {capm_result['beta']:>6.3f} "
          f"{capm_result['alpha_annual']:>11.2%} "
          f"{capm_result['r_squared']:>6.3f} "
          f"{is_sig:>13}")